# Preprocess Data

Align AlphaEarth features and the coarse population grid, then save the
processed arrays used by the modelling notebooks.

**Inputs** (place in `data/raw/` before running):
- `data/raw/alphaearth_features.npy` — shape `(100, 100, 64)`
- `data/raw/population_coarse.npy`  — shape `(10, 10)`

**Outputs** (written to `data/processed/`):
- `ae_fine.npy`   — shape `(10000, 64)`
- `y_coarse.npy`  — shape `(100,)`
- `block_ids.npy` — shape `(10000,)`

In [ ]:
from pathlib import Path
import numpy as np


ROOT = Path("..").resolve() if Path.cwd().name == "data" else Path(".").resolve()
DATA_RAW = ROOT / "data" / "raw"
DATA_PROCESSED = ROOT / "data" / "processed"
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

FINE_H = 100
FINE_W = 100
COARSE_H = 10
COARSE_W = 10
AE_DIM = 64
FINE_PER_COARSE_SIDE = FINE_H // COARSE_H

print("Root:", ROOT)

In [ ]:
ae = np.load(DATA_RAW / "alphaearth_features.npy")          # (100, 100, 64)
y_coarse_grid = np.load(DATA_RAW / "population_coarse.npy") # (10, 10)

assert ae.shape == (FINE_H, FINE_W, AE_DIM), f"Unexpected AE shape: {ae.shape}"
assert y_coarse_grid.shape == (COARSE_H, COARSE_W), (
    f"Unexpected coarse population shape: {y_coarse_grid.shape}"
)

print("AE shape          :", ae.shape)
print("Coarse pop shape  :", y_coarse_grid.shape)

In [ ]:
# Flatten fine cells
ae_fine = ae.reshape(-1, AE_DIM).astype(np.float32)

# Flatten coarse targets
y_coarse = y_coarse_grid.reshape(-1).astype(np.float32)

# Block IDs: map each fine cell to its coarse block index
block_ids_grid = np.zeros((FINE_H, FINE_W), dtype=np.int64)
for r in range(FINE_H):
    for c in range(FINE_W):
        coarse_r = r // FINE_PER_COARSE_SIDE
        coarse_c = c // FINE_PER_COARSE_SIDE
        block_ids_grid[r, c] = coarse_r * COARSE_W + coarse_c

block_ids = block_ids_grid.reshape(-1)

print("ae_fine    :", ae_fine.shape)
print("y_coarse   :", y_coarse.shape)
print("block_ids  :", block_ids.shape)
print("Cells/block:", np.bincount(block_ids)[:5])

In [ ]:
np.save(DATA_PROCESSED / "ae_fine.npy", ae_fine)
np.save(DATA_PROCESSED / "y_coarse.npy", y_coarse)
np.save(DATA_PROCESSED / "block_ids.npy", block_ids)

print("Saved processed arrays to", DATA_PROCESSED)